# Non-Food Cohorts Push Module

Mirrors price + cart rule changes from any source module (M2, M3, M4, M5) to a new set of **non-food cohorts** (one per existing region).

## Visibility Logic
- **Non-food SKUs** (where `categories.section_id` is in the non-food set): visible (with min-PU hide rule from `disable_pu_visibility` sheet)
- **Food SKUs**: customized but **invisible for ALL PUs** (so they don't fall back to the parent cohort and become orderable)

## Custom push method
This module has its own push method (does NOT reuse `push_prices()` or `push_cart_rules()`). It builds the upload Excel files itself with explicit `Visibility (YES/NO)` per row, then calls the low-level `post_prices()` and `post_cart_rules()` API helpers from `push_prices_handler` and `push_cart_rules_handler`.

## Usage

```python
%run non_food_cohorts_push.ipynb

# After Module 2/3/4 push (warehouse-level df with new_price/new_cart_rule):
result = push_to_non_food_cohorts(df_output, source_module='module_2', mode='live')
send_summary_to_slack(result)

# After Module 5 push (cohort-level PU-level df with price):
result = push_to_non_food_cohorts(new_intros, source_module='module_5', mode='live')
send_summary_to_slack(result)
```

In [ ]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================
import pandas as pd
import numpy as np
import os
import sys
import tempfile
from datetime import datetime
import pytz

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))
sys.path.insert(0, os.path.abspath('.'))

import setup_environment_2
setup_environment_2.initialize_env()
from db import query_snowflake
from common_functions import send_text_slack, send_file_slack

# Load API helpers (post_prices, post_cart_rules) and disable_pu_visibility loader
%run push_prices_handler.ipynb
%run push_cart_rules_handler.ipynb
%run queries_module.ipynb

CAIRO_TZ = pytz.timezone('Africa/Cairo')

print("Non-food cohorts push module: imports loaded")

In [ ]:
# =============================================================================
# CONSTANTS
# =============================================================================

# Section IDs that classify a product as non-food
NON_FOOD_SECTION_IDS = {714, 626, 516, 417, 351, 121, 285, 20, 54, 37, 10, 43, 44, 36, 14, 8, 55, 619, 622, 621}

# Mapping from existing cohort IDs to new non-food cohort IDs.
# List of (source_cohort_id, non_food_cohort_id) tuples — supports multiple
# non-food cohorts mapped to the same source cohort (e.g. two Cairo cohorts
# both inheriting from 700). Rows from the source module will be duplicated
# for each matching new cohort.
NON_FOOD_COHORT_MAP = [
    (700, 1255),   # Cairo - secondary
    (700, 1288),   # Cairo
    (701, 1289),   # Giza
    (702, 1290),   # Alexandria
    (703, 1291),   # Delta West
    (704, 1292),   # Delta East
    (1123, 1293),  # Upper Egypt - Menya
    (1124, 1294),  # Upper Egypt - Assiut
    (1125, 1295),  # Upper Egypt - Sohag
    (1126, 1296),  # Upper Egypt - Beni Suef
]

# Source cohort IDs we care about (for filtering input rows before expansion)
SOURCE_COHORT_IDS = {src for src, _ in NON_FOOD_COHORT_MAP}

SLACK_CHANNEL_NAME = 'new-pricing-logic'
SLACK_CHANNEL_ID = 'C0AAWK97Z3Q'

# Cart rule bounds (matches push_cart_rules_handler defaults)
MIN_CART_RULE = 2
MAX_CART_RULE = 150

# Upload chunk sizes (matches push_prices_handler / push_cart_rules_handler)
CHUNK_SIZE_DEFAULT = 4000
CHUNK_SIZE_SPECIAL = 2000
# Cohorts that need smaller chunks (override default). Add cohort IDs here if needed.
CHUNK_SIZE_OVERRIDES = {}

print(f"Non-food sections: {len(NON_FOOD_SECTION_IDS)} | Cohort mappings: {len(NON_FOOD_COHORT_MAP)} (source cohorts: {len(SOURCE_COHORT_IDS)})")

In [ ]:
# =============================================================================
# SECTION LOOKUP + SAFETY CHECK QUERY
# =============================================================================

def get_product_sections():
    """Return product_id -> section_id mapping from products + categories."""
    q = """
    SELECT p.id AS product_id, c.section_id
    FROM products p
    JOIN categories c ON c.id = p.category_id
    """
    df = query_snowflake(q)
    df.columns = df.columns.str.lower()
    df['product_id'] = pd.to_numeric(df['product_id'], errors='coerce').astype('Int64')
    df['section_id'] = pd.to_numeric(df['section_id'], errors='coerce').astype('Int64')
    return df.dropna(subset=['product_id'])


def get_visible_food_skus_on_non_food_cohorts():
    """Find food SKUs effectively visible on the new non-food cohorts.

    Effective visibility = if the new cohort row is_customized then its is_visible,
    otherwise fall back to the source cohort's is_visible.

    Returns a df with: cohort_id, product_id, packing_unit_id, pu_price
    where pu_price is the price the SKU has in its source cohort.
    """
    cohort_pairs = ", ".join(f"({src},{dst})" for src, dst in NON_FOOD_COHORT_MAP)
    sections = ", ".join(str(s) for s in NON_FOOD_SECTION_IDS)
    q = f"""
    WITH non_food_cohorts AS (
        SELECT * FROM (VALUES {cohort_pairs}) v(source_cohort_id, non_food_cohort_id)
    ),
    visibility_chain AS (
        SELECT
            nfc.non_food_cohort_id AS cohort_id,
            ppu.product_id,
            ppu.packing_unit_id,
            COALESCE(
                CASE WHEN cppu_new.is_customized = TRUE THEN cppu_new.is_visible END,
                cppu_src.is_visible
            ) AS effective_visible,
            cppu_src.price AS source_price
        FROM non_food_cohorts nfc
        CROSS JOIN packing_unit_products ppu
        LEFT JOIN cohort_product_packing_units cppu_new
            ON cppu_new.cohort_id = nfc.non_food_cohort_id
            AND cppu_new.product_packing_unit_id = ppu.id
        LEFT JOIN cohort_product_packing_units cppu_src
            ON cppu_src.cohort_id = nfc.source_cohort_id
            AND cppu_src.product_packing_unit_id = ppu.id
    )
    SELECT vc.cohort_id, vc.product_id, vc.packing_unit_id,
           vc.source_price AS pu_price
    FROM visibility_chain vc
    JOIN products p ON p.id = vc.product_id
    JOIN categories c ON c.id = p.category_id
    WHERE vc.effective_visible = TRUE
      AND vc.source_price IS NOT NULL
      AND vc.source_price > 0
      AND c.section_id NOT IN ({sections})
    """
    df = query_snowflake(q)
    if df is None or len(df) == 0:
        return pd.DataFrame(columns=['cohort_id', 'product_id', 'packing_unit_id', 'pu_price'])
    df.columns = df.columns.str.lower()
    for col in ['cohort_id', 'product_id', 'packing_unit_id']:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
    df['pu_price'] = pd.to_numeric(df['pu_price'], errors='coerce')
    return df.dropna(subset=['cohort_id', 'product_id', 'packing_unit_id', 'pu_price'])


print("get_product_sections() defined")
print("get_visible_food_skus_on_non_food_cohorts() defined")

In [ ]:
# =============================================================================
# INPUT NORMALIZATION
# =============================================================================
# Different source modules pass different shapes:
#   - M2/M3/M4: warehouse-level rows with product_id, warehouse_id, cohort_id,
#               stocks, current_price, new_price, current_cart_rule, new_cart_rule
#               (new_price is per BASIC unit)
#   - M5:       cohort-level PU-level rows with cohort_id, product_id,
#               packing_unit_id, sku, brand, cat, price (per PU)
#
# normalize_input() produces a common shape:
#   columns = product_id, cohort_id, packing_unit_id, basic_unit_count,
#             pu_price, cart_rule
# =============================================================================

def _normalize_columns(df):
    """Lower-case columns to handle Snowflake's uppercase quirk."""
    df = df.copy()
    df.columns = [c.lower() if isinstance(c, str) else c for c in df.columns]
    return df


def _is_m5_shape(df):
    """M5 shape has packing_unit_id + (price or final_new_price), no new_price column."""
    cols = set(df.columns)
    has_m5_price = ('price' in cols) or ('final_new_price' in cols)
    return 'packing_unit_id' in cols and has_m5_price and 'new_price' not in cols


def normalize_input(df_input, pus):
    """Normalize any source df into a common shape:
       (product_id, cohort_id, packing_unit_id, basic_unit_count, pu_price, cart_rule)
    """
    df = _normalize_columns(df_input)
    pus = _normalize_columns(pus)

    if _is_m5_shape(df):
        price_col = 'final_new_price' if 'final_new_price' in df.columns else 'price'
        cols_needed = ['cohort_id', 'product_id', 'packing_unit_id', price_col]
        if 'sku' in df.columns:
            cols_needed.append('sku')
        out = df[cols_needed].copy()
        out = out.rename(columns={price_col: 'pu_price'})
        out = out.merge(
            pus[['product_id', 'packing_unit_id', 'basic_unit_count']],
            on=['product_id', 'packing_unit_id'], how='left'
        )
        out['basic_unit_count'] = pd.to_numeric(out['basic_unit_count'], errors='coerce').fillna(1)
        out['cart_rule'] = np.nan
        out['pu_price'] = pd.to_numeric(out['pu_price'], errors='coerce')
        out = out.dropna(subset=['pu_price'])
        out = out[out['pu_price'] > 0]  # drop zero/negative prices (MaxAB rejects)
        if 'sku' not in out.columns:
            out['sku'] = out['product_id'].astype(str)
        return out

    # M2/M3/M4 shape: warehouse-level basic-unit prices
    required = ['product_id', 'cohort_id', 'new_price']
    for col in required:
        if col not in df.columns:
            raise ValueError(f"Required column '{col}' missing from input df")

    if 'stocks' not in df.columns:
        df['stocks'] = 1
    df['stocks'] = pd.to_numeric(df['stocks'], errors='coerce').fillna(1)
    df['new_price'] = pd.to_numeric(df['new_price'], errors='coerce')

    # Filter to rows with a new price
    df = df.dropna(subset=['new_price'])
    if 'current_price' in df.columns:
        df['current_price'] = pd.to_numeric(df['current_price'], errors='coerce')
        df = df[df['new_price'] != df['current_price']]

    if df.empty:
        return pd.DataFrame(columns=['product_id', 'cohort_id', 'packing_unit_id',
                                      'basic_unit_count', 'pu_price', 'cart_rule'])

    # Stock-weighted average price + first sku per (product_id, cohort_id)
    grp = df.groupby(['product_id', 'cohort_id'], as_index=False)
    has_sku = 'sku' in df.columns
    def _wavg(g):
        w = g['stocks'].clip(lower=0)
        wsum = w.sum()
        out = {}
        if wsum > 0:
            out['basic_price'] = (g['new_price'] * w).sum() / wsum
        else:
            out['basic_price'] = g['new_price'].mean()
        if has_sku:
            first_sku = g['sku'].dropna().astype(str)
            out['sku'] = first_sku.iloc[0] if len(first_sku) > 0 else ''
        return pd.Series(out)
    basic = grp.apply(_wavg).reset_index()
    keep_cols = ['product_id', 'cohort_id', 'basic_price']
    if has_sku:
        keep_cols.append('sku')
    basic = basic[keep_cols]

    # Cart rule = min across warehouses (matches push_cart_rules_handler)
    if 'new_cart_rule' in df.columns:
        df['new_cart_rule'] = pd.to_numeric(df['new_cart_rule'], errors='coerce')
        cart = df.dropna(subset=['new_cart_rule']).groupby(
            ['product_id', 'cohort_id'], as_index=False
        )['new_cart_rule'].min().rename(columns={'new_cart_rule': 'cart_rule'})
        basic = basic.merge(cart, on=['product_id', 'cohort_id'], how='left')
    else:
        basic['cart_rule'] = np.nan

    # Expand to all PUs of each product
    expanded = basic.merge(
        pus[['product_id', 'packing_unit_id', 'basic_unit_count']],
        on='product_id', how='inner'
    )
    expanded['basic_unit_count'] = pd.to_numeric(expanded['basic_unit_count'], errors='coerce').fillna(1)
    expanded['pu_price'] = (expanded['basic_price'] * expanded['basic_unit_count']).round(2)
    expanded = expanded[expanded['pu_price'] > 0]  # drop zero/negative prices (MaxAB rejects)
    if 'sku' not in expanded.columns:
        expanded['sku'] = expanded['product_id'].astype(str)
    return expanded[['product_id', 'cohort_id', 'packing_unit_id',
                     'basic_unit_count', 'pu_price', 'cart_rule', 'sku']]


print("normalize_input() defined")

In [ ]:
# =============================================================================
# CUSTOM PUSH METHOD
# =============================================================================
# Builds Excel upload files per cohort with explicit Visibility column,
# then calls post_prices() and post_cart_rules() (low-level API helpers
# from push_prices_handler / push_cart_rules_handler) per cohort.
# =============================================================================

def _set_visibility(df_norm):
    """Set Visibility (YES/NO) per row based on is_non_food + min-PU disable rule."""
    df_norm = df_norm.sort_values(['cohort_id', 'product_id', 'basic_unit_count']).copy()
    df_norm['min_pu_flag'] = df_norm.groupby(['cohort_id', 'product_id'])['basic_unit_count'].rank(method='first').eq(1).astype(int)

    # Load disable_pu_visibility list (same source as push_prices_handler)
    try:
        disable_min = get_disable_pu_visibility()
        disable_set = set(pd.to_numeric(disable_min['product_id'], errors='coerce').dropna().astype(int).unique())
    except Exception as e:
        print(f"Warning: could not load disable_pu_visibility ({e}); proceeding with no min-PU hide")
        disable_set = set()

    def _vis(row):
        # Food: invisible for ALL PUs
        if not row['is_non_food']:
            return 'NO'
        # Non-food + smallest PU + product flagged for hide
        if row['min_pu_flag'] == 1 and int(row['product_id']) in disable_set:
            return 'NO'
        # Hardcoded exception (same as push_prices_handler)
        if int(row['product_id']) == 1309 and int(row['packing_unit_id']) == 23:
            return 'NO'
        return 'YES'

    df_norm['Visibility'] = df_norm.apply(_vis, axis=1)
    return df_norm


def _build_price_excel(group, file_path):
    """Write a price upload Excel for one cohort.
    Product Name must be a non-empty string — MaxAB API rejects NaN/empty.
    Uses .values to strip Series/Int64 metadata that can cause phantom columns.
    Format mirrors module_5: to_excel(..., index=False, engine='xlsxwriter')."""
    if 'sku' in group.columns:
        sku_col = group['sku'].fillna('').astype(str)
        sku_col = sku_col.where(sku_col.str.len() > 0, group['product_id'].astype(str))
    else:
        sku_col = group['product_id'].astype(str)

    payload = pd.DataFrame({
        'Product ID': group['product_id'].astype(int).values,
        'Product Name': sku_col.astype(str).values,
        'Packing Unit ID': group['packing_unit_id'].astype(int).values,
        'Price': group['pu_price'].astype(float).round(2).values,
        'Visibility (YES/NO)': group['Visibility'].astype(str).values,
    })
    # Final safety net: Product Name must never be NaN or empty
    payload['Product Name'] = payload['Product Name'].fillna('').astype(str)
    payload.loc[payload['Product Name'].str.len() == 0, 'Product Name'] = payload['Product ID'].astype(str)
    payload['Execute At (format:dd/mm/yyyy HH:mm)'] = None
    payload['Tags'] = None
    payload.to_excel(file_path, index=False)


def _build_cart_excel(group, file_path):
    """Write a cart rule upload Excel for one cohort.
    Uses .values to strip Series/Int64 metadata that can cause phantom columns.
    Format mirrors module_5: to_excel(..., index=False, engine='xlsxwriter')."""
    payload = pd.DataFrame({
        'Product ID': group['product_id'].astype(int).values,
        'Packing Unit ID': group['packing_unit_id'].astype(int).values,
        'Cart Rules': group['cart_rule_clean'].astype(int).values,
    })
    payload.to_excel(file_path, index=False)


def custom_push(df_norm, source_module, mode):
    """Push prices + cart rules per cohort. Builds Excels, calls API helpers.

    Args:
        df_norm: normalized df with columns
                 product_id, cohort_id, packing_unit_id, basic_unit_count,
                 pu_price, cart_rule, is_non_food (+ optional 'sku')
        source_module: tag for filenames / logging
        mode: 'testing' (build files only, no upload) or 'live' (upload to API)

    Returns:
        dict with pushed_cohorts, failed_cohorts, file_paths, rows
    """
    if df_norm.empty:
        return {'pushed_prices_cohorts': [], 'failed_prices_cohorts': [],
                'pushed_carts_cohorts': [], 'failed_carts_cohorts': [],
                'file_paths': [], 'rows': 0}

    # Drop rows with missing critical fields (would crash the per-cohort upload)
    before = len(df_norm)
    df_norm = df_norm.dropna(subset=['cohort_id', 'product_id', 'packing_unit_id', 'pu_price']).copy()
    # Also drop zero or negative prices (MaxAB API rejects these with validation.500)
    df_norm = df_norm[df_norm['pu_price'] > 0]
    # Drop duplicate (cohort, product, PU) rows keeping first
    df_norm = df_norm.drop_duplicates(subset=['cohort_id', 'product_id', 'packing_unit_id'], keep='first')
    if len(df_norm) < before:
        print(f"  Dropped {before - len(df_norm)} rows with missing/invalid data (NaN, zero prices, or duplicates)")
    if df_norm.empty:
        return {'pushed_prices_cohorts': [], 'failed_prices_cohorts': [],
                'pushed_carts_cohorts': [], 'failed_carts_cohorts': [],
                'file_paths': [], 'rows': 0}

    df_norm = _set_visibility(df_norm)

    # Cart rule cleanup: convert larger PUs from basic units to packing units, clip to bounds
    df_norm['cart_rule_clean'] = df_norm.apply(
        lambda r: max(MIN_CART_RULE, min(MAX_CART_RULE, int(round((r['cart_rule'] or 0) / max(r['basic_unit_count'], 1)))))
        if pd.notna(r['cart_rule']) and r['cart_rule'] > 0
        else None,
        axis=1
    )

    pushed_prices, failed_prices = [], []
    pushed_carts, failed_carts = [], []
    file_paths = []

    out_dir = tempfile.gettempdir()
    timestamp = datetime.now(CAIRO_TZ).strftime('%Y%m%d_%H%M%S')

    def _chunk_size_for(cohort_int):
        """Return chunk size for a cohort (override > special > default)."""
        if cohort_int in CHUNK_SIZE_OVERRIDES:
            return CHUNK_SIZE_OVERRIDES[cohort_int]
        return CHUNK_SIZE_DEFAULT

    for cohort_id, group in df_norm.groupby('cohort_id'):
        cohort_int = int(cohort_id)
        chunk_size = _chunk_size_for(cohort_int)

        # ---- Prices ----
        n_price_chunks = (len(group) + chunk_size - 1) // chunk_size
        price_chunk_results = []
        for i in range(n_price_chunks):
            chunk_df = group.iloc[i * chunk_size:(i + 1) * chunk_size]
            price_file = os.path.join(
                out_dir,
                f'nonfood_prices_{source_module}_{cohort_int}_chunk{i+1}of{n_price_chunks}_{timestamp}.xlsx'
            )
            _build_price_excel(chunk_df, price_file)
            file_paths.append(price_file)

            if mode == 'live':
                try:
                    resp = post_prices(cohort_int, price_file)
                    if resp.status_code == 200 and b'"success":true' in resp.content:
                        price_chunk_results.append(True)
                        print(f"  Cohort {cohort_int} prices chunk {i+1}/{n_price_chunks} ({len(chunk_df)} rows): OK")
                    else:
                        price_chunk_results.append(False)
                        print(f"  Cohort {cohort_int} prices chunk {i+1}/{n_price_chunks}: status {resp.status_code} | {resp.content[:200]}")
                except Exception as e:
                    price_chunk_results.append(False)
                    print(f"  Cohort {cohort_int} prices chunk {i+1}/{n_price_chunks} error: {e}")
            else:
                price_chunk_results.append(True)
                print(f"  [TESTING] Cohort {cohort_int} prices chunk {i+1}/{n_price_chunks}: {price_file} ({len(chunk_df)} rows)")

        # Cohort counted as "pushed" only if ALL chunks succeeded
        if price_chunk_results and all(price_chunk_results):
            pushed_prices.append(cohort_int)
        else:
            failed_prices.append(cohort_int)

        # ---- Cart rules (only push rows with a cart rule value) ----
        cart_group = group.dropna(subset=['cart_rule_clean'])
        if cart_group.empty:
            continue

        n_cart_chunks = (len(cart_group) + chunk_size - 1) // chunk_size
        cart_chunk_results = []
        for i in range(n_cart_chunks):
            chunk_df = cart_group.iloc[i * chunk_size:(i + 1) * chunk_size]
            cart_file = os.path.join(
                out_dir,
                f'nonfood_carts_{source_module}_{cohort_int}_chunk{i+1}of{n_cart_chunks}_{timestamp}.xlsx'
            )
            _build_cart_excel(chunk_df, cart_file)
            file_paths.append(cart_file)

            if mode == 'live':
                try:
                    resp = post_cart_rules(cohort_int, cart_file)
                    if resp.status_code == 200 and b'Cart Rules Updated Successfully' in resp.content:
                        cart_chunk_results.append(True)
                        print(f"  Cohort {cohort_int} cart chunk {i+1}/{n_cart_chunks} ({len(chunk_df)} rows): OK")
                    else:
                        cart_chunk_results.append(False)
                        print(f"  Cohort {cohort_int} cart chunk {i+1}/{n_cart_chunks}: status {resp.status_code} | {resp.content[:200]}")
                except Exception as e:
                    cart_chunk_results.append(False)
                    print(f"  Cohort {cohort_int} cart chunk {i+1}/{n_cart_chunks} error: {e}")
            else:
                cart_chunk_results.append(True)
                print(f"  [TESTING] Cohort {cohort_int} cart chunk {i+1}/{n_cart_chunks}: {cart_file} ({len(chunk_df)} rows)")

        if cart_chunk_results and all(cart_chunk_results):
            pushed_carts.append(cohort_int)
        else:
            failed_carts.append(cohort_int)

    return {
        'pushed_prices_cohorts': pushed_prices,
        'failed_prices_cohorts': failed_prices,
        'pushed_carts_cohorts': pushed_carts,
        'failed_carts_cohorts': failed_carts,
        'file_paths': file_paths,
        'rows': len(df_norm),
    }


print("custom_push() defined")

In [ ]:
# =============================================================================
# MAIN ENTRY POINT
# =============================================================================

def push_to_non_food_cohorts(df_input, source_module='unknown', mode='testing'):
    """
    Mirror price + cart rule changes from a source module to the non-food cohorts.

    Args:
        df_input: df from M2/M3/M4 (warehouse-level basic-unit prices) or
                  M5 (cohort-level PU-level prices)
        source_module: tag for filenames / Slack
        mode: 'testing' (build files only) or 'live' (upload to API)

    Returns:
        dict with rows pushed, food/non-food counts, pushed/failed cohorts
    """
    print(f"\n{'='*60}")
    print(f"NON-FOOD COHORTS PUSH | source: {source_module} | mode: {mode}")
    print(f"{'='*60}")

    def _empty_result(reason):
        print(reason)
        return {'source_module': source_module, 'mode': mode, 'rows': 0,
                'food_count': 0, 'non_food_count': 0, 'safety_check_count': 0,
                'pushed_prices_cohorts': [], 'failed_prices_cohorts': [],
                'pushed_carts_cohorts': [], 'failed_carts_cohorts': [],
                'file_paths': []}

    if df_input is None or len(df_input) == 0:
        return _empty_result("No input rows. Skipping.")

    # Step 1: load packing units lookup
    pus = get_packing_units()

    # Step 2: normalize input shape
    df_norm = normalize_input(df_input, pus)
    if df_norm.empty:
        return _empty_result("Normalized df is empty (no price changes to mirror). Skipping.")

    # Step 3: keep only rows in source cohorts we have a mapping for
    df_norm = df_norm[df_norm['cohort_id'].isin(SOURCE_COHORT_IDS)].copy()
    if df_norm.empty:
        return _empty_result(f"No rows in source cohorts {sorted(SOURCE_COHORT_IDS)}. Skipping.")

    # Step 4: remap cohort_id to the non-food cohort IDs.
    # Supports multiple new cohorts per source (list-of-tuples). Rows are duplicated
    # via merge for each matching (source, new) pair.
    mapping_df = pd.DataFrame(NON_FOOD_COHORT_MAP, columns=['source_cohort_id', 'cohort_id_new'])
    df_norm = df_norm.rename(columns={'cohort_id': 'source_cohort_id'}).merge(
        mapping_df, on='source_cohort_id', how='inner'
    )
    df_norm = df_norm.rename(columns={'cohort_id_new': 'cohort_id'})

    # Step 5: classify food vs non-food
    sections = get_product_sections()
    df_norm = df_norm.merge(sections, on='product_id', how='left')
    df_norm['is_non_food'] = df_norm['section_id'].apply(
        lambda s: pd.notna(s) and int(s) in NON_FOOD_SECTION_IDS
    )

    food_count = int((~df_norm['is_non_food']).sum())
    non_food_count = int(df_norm['is_non_food'].sum())
    print(f"  Total rows: {len(df_norm)}")
    print(f"  Non-food (visible): {non_food_count} rows")
    print(f"  Food (customized invisible): {food_count} rows")
    print(f"  Cohorts: {sorted(df_norm['cohort_id'].unique().tolist())}")

    # Step 6: safety check - find food SKUs effectively visible on the new cohorts
    # (either explicitly customized as visible, or inherited visibility from source cohort).
    # Force-customize them as Visibility = NO using the price from their source cohort.
    print("\nRunning safety check for visible food SKUs on non-food cohorts...")
    safety_count = 0
    try:
        df_safety = get_visible_food_skus_on_non_food_cohorts()
        print(f"  Found {len(df_safety)} food PUs effectively visible on non-food cohorts")

        if len(df_safety) > 0:
            df_safety = df_safety.merge(
                pus[['product_id', 'packing_unit_id', 'basic_unit_count']],
                on=['product_id', 'packing_unit_id'], how='left'
            )
            df_safety['basic_unit_count'] = pd.to_numeric(df_safety['basic_unit_count'], errors='coerce').fillna(1)
            df_safety['cart_rule'] = np.nan
            # Each non-food cohort_id has exactly one source cohort, so reverse is safe
            inv_map = {dst: src for src, dst in NON_FOOD_COHORT_MAP}
            df_safety['source_cohort_id'] = df_safety['cohort_id'].map(inv_map)
            df_safety['section_id'] = pd.NA
            df_safety['is_non_food'] = False
            # sku fallback (safety-check query doesn't pull SKU names)
            df_safety['sku'] = df_safety['product_id'].astype(str)

            # Drop rows already present in df_norm to avoid duplicate uploads
            existing_keys = set(zip(
                df_norm['cohort_id'].astype('Int64'),
                df_norm['product_id'].astype('Int64'),
                df_norm['packing_unit_id'].astype('Int64')
            ))
            df_safety = df_safety[~df_safety.apply(
                lambda r: (r['cohort_id'], r['product_id'], r['packing_unit_id']) in existing_keys,
                axis=1
            )]

            if len(df_safety) > 0:
                # Align columns and concat
                df_safety = df_safety[df_norm.columns.tolist()]
                df_norm = pd.concat([df_norm, df_safety], ignore_index=True)
                safety_count = len(df_safety)
                print(f"  Added {safety_count} safety-check rows (food, force-invisible)")
            else:
                print("  All safety-check rows already in payload - no additions needed")
    except Exception as e:
        print(f"  Warning: safety check failed ({e}); proceeding without it")

    # Step 7: push (build files + upload)
    push_result = custom_push(df_norm, source_module, mode)

    result = {
        'source_module': source_module,
        'mode': mode,
        'rows': push_result['rows'],
        'food_count': food_count,
        'non_food_count': non_food_count,
        'safety_check_count': safety_count,
        'pushed_prices_cohorts': push_result['pushed_prices_cohorts'],
        'failed_prices_cohorts': push_result['failed_prices_cohorts'],
        'pushed_carts_cohorts': push_result['pushed_carts_cohorts'],
        'failed_carts_cohorts': push_result['failed_carts_cohorts'],
        'file_paths': push_result['file_paths'],
    }

    print(f"\n{'='*60}")
    print(f"DONE | prices pushed: {len(result['pushed_prices_cohorts'])}, failed: {len(result['failed_prices_cohorts'])}")
    print(f"     | carts  pushed: {len(result['pushed_carts_cohorts'])}, failed: {len(result['failed_carts_cohorts'])}")
    print(f"{'='*60}\n")
    return result


print("push_to_non_food_cohorts() defined")

In [ ]:
# =============================================================================
# SLACK NOTIFICATION
# =============================================================================

def send_summary_to_slack(result):
    """Send a Slack summary of the push result."""
    timestamp = datetime.now(CAIRO_TZ).strftime('%Y-%m-%d %H:%M')
    failed = (len(result.get('failed_prices_cohorts', [])) +
              len(result.get('failed_carts_cohorts', [])))
    status = 'Completed' if failed == 0 else 'Completed with failures'

    msg = (
        f"*Non-Food Cohorts Push: {status}*\n"
        f"\n"
        f"*Source:* `{result.get('source_module', 'unknown')}`\n"
        f"*Mode:* `{result.get('mode', 'unknown')}`\n"
        f"*Time:* {timestamp} Cairo\n"
        f"\n"
        f"*Rows:* {result.get('rows', 0)}\n"
        f"*Non-food (visible):* {result.get('non_food_count', 0)}\n"
        f"*Food (customized invisible):* {result.get('food_count', 0)}\n"
        f"*Safety check (food force-fixed):* {result.get('safety_check_count', 0)}\n"
        f"\n"
        f"*Prices:* pushed {len(result.get('pushed_prices_cohorts', []))}, "
        f"failed {len(result.get('failed_prices_cohorts', []))}\n"
        f"*Cart rules:* pushed {len(result.get('pushed_carts_cohorts', []))}, "
        f"failed {len(result.get('failed_carts_cohorts', []))}\n"
    )
    if result.get('failed_prices_cohorts') or result.get('failed_carts_cohorts'):
        msg += (
            f"\n*Failed price cohorts:* {result.get('failed_prices_cohorts', [])}\n"
            f"*Failed cart cohorts:* {result.get('failed_carts_cohorts', [])}\n"
        )

    try:
        send_text_slack(SLACK_CHANNEL_NAME, msg)
        print("Slack summary sent")
    except Exception as e:
        print(f"Failed to send Slack summary: {e}")


print("send_summary_to_slack() defined")
print("\nModule ready. Use:")
print("  result = push_to_non_food_cohorts(df, source_module='module_X', mode='testing')")
print("  send_summary_to_slack(result)")